# 15 - Random Forest: validation choice and hybrid

1. Choose the imbalance method on a validation part instead of the test set.
2. Hybrid: Random Forest OR the unseen-event rule from notebook 14.
3. JSON explanations of flagged windows for the dashboard.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import evaluate
from src.models.supervised import example_alerts, hybrid_predict, save_alerts, select_rf_method
from src.models.unsupervised import has_unseen, unseen_columns

DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
DATASETS = {"HDFS": ("hdfs", "order", "BlockId"), "BGL": ("bgl", "window", "window")}

data = {name: (pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet"),
               pd.read_csv(PROCESSED / f"{prefix}_templates.csv").set_index("EventId")["template"])
        for name, (prefix, _, _) in DATASETS.items()}

## Validation choice

Each method ('none', 'balanced', 'smote') is fitted on the first 60% (all labels), the method with the best F1 on the next 10% is chosen, and only that one is evaluated on the last 30% (the same test set as before).

In [2]:
results = {name: select_rf_method(data[name][0], order_col=order_col)
           for name, (_, order_col, _) in DATASETS.items()}

pd.concat({name: r["val"] for name, r in results.items()}, names=["dataset"]).reset_index(level=0)[
    ["dataset", "method", "precision", "recall", "f1", "false_alarm_rate"]].style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

dataset,method,precision,recall,f1,false_alarm_rate
HDFS,none,0.9990,0.9980,0.9985,0.0000
HDFS,balanced,0.9990,0.9985,0.9987,0.0000
HDFS,smote,0.9889,0.9980,0.9934,0.0004
BGL,none,0.9851,0.0942,0.1719,0.0001
BGL,balanced,0.9756,0.1141,0.2043,0.0003
BGL,smote,0.9832,0.1113,0.1999,0.0002


Test results for the chosen method:

In [3]:
chosen = pd.DataFrame([{"dataset": name, **r["test"]} for name, r in results.items()])
chosen[["dataset", "method", "val_f1", "precision", "recall", "f1", "false_alarm_rate"]].style.format(
    {c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]} | {"val_f1": "{:.4f}"}).hide(axis="index")

dataset,method,val_f1,precision,recall,f1,false_alarm_rate
HDFS,balanced,0.9987,1.0000,0.9970,0.9985,0.0000
BGL,balanced,0.2043,0.9551,0.0721,0.1341,0.0003


Old tables (notebooks 07 and 11: fitted on the full 70%, every method evaluated on the test set), for comparison:

In [4]:
old = pd.concat({name: pd.read_csv(DOCS / f"{prefix}_rf_results.csv") for name, (prefix, _, _) in DATASETS.items()},
                names=["dataset"]).reset_index(level=0)
old[["dataset", "method", "precision", "recall", "f1", "false_alarm_rate"]].style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]}).hide(axis="index")

dataset,method,precision,recall,f1,false_alarm_rate
HDFS,No imbalance handling,1.0000,0.9984,0.9992,0.0000
HDFS,class_weight='balanced',1.0000,0.9984,0.9992,0.0000
HDFS,SMOTE,0.9655,0.9989,0.9819,0.0008
BGL,No imbalance handling,0.9951,0.5348,0.6957,0.0002
BGL,class_weight='balanced',0.9934,0.5365,0.6967,0.0003
BGL,SMOTE,0.9944,0.5373,0.6976,0.0002


## Hybrid: Random Forest OR unseen event

A window is flagged if the Random Forest says anomaly or it has an event that never occurs in the 60% fit part (`unseen_columns` / `has_unseen`, the same rule as `IForestDetector(flag_unseen=True)`).

In [5]:
rows, unseen_masks = [], {}
for name, r in results.items():
    fit, _, test = r["parts"]
    cols = r["event_cols"]
    unseen_masks[name] = unseen = unseen_columns(fit[cols])
    X_test, y_test = test[cols], test["Label"].to_numpy()
    new = has_unseen(X_test, unseen)
    for label, pred in [("Random Forest", r["model"].predict(X_test)), ("hybrid", hybrid_predict(r["model"], X_test, unseen))]:
        m = evaluate(y_test, pred)
        rows.append({"dataset": name, "model": label, "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
                     "false_alarm_rate": m["false_alarm_rate"],
                     "recall, windows with unseen event": pred[(y_test == 1) & new].mean()})
    print(f"{name}: {unseen.sum()} of {len(cols)} events unseen in the fit part; test windows with one: "
          f"anomalous {new[y_test == 1].sum():,} of {(y_test == 1).sum():,}, normal {new[y_test == 0].sum():,} of {(y_test == 0).sum():,}")

hybrid = pd.DataFrame(rows)
hybrid.round(4).to_csv(DOCS / "rf_validated_hybrid_results.csv", index=False)
hybrid.style.format({c: "{:.4f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]} | {"recall, windows with unseen event": "{:.3f}"}).hide(axis="index")

HDFS: 2 of 45 events unseen in the fit part; test windows with one: anomalous 6 of 3,723, normal 0 of 168,796


BGL: 171 of 391 events unseen in the fit part; test windows with one: anomalous 4,254 of 5,312, normal 47,266 of 65,908


dataset,model,precision,recall,f1,false_alarm_rate,"recall, windows with unseen event"
HDFS,Random Forest,1.0000,0.9970,0.9985,0.0000,0.667
HDFS,hybrid,1.0000,0.9976,0.9988,0.0000,1.000
BGL,Random Forest,0.9551,0.0721,0.1341,0.0003,0.041
BGL,hybrid,0.0862,0.8400,0.1564,0.7174,1.000


## Example alerts for the dashboard

20 flagged test windows per dataset: up to 5 flagged only by the unseen-event rule, the rest by the Random Forest. Model alerts list the 3 events with the largest SHAP value; unseen-event alerts list the unseen templates (no SHAP value). BGL uses an interventional SHAP background of 100 fit windows, as in notebook 11.

In [6]:
alerts = {}
for name, (prefix, _, id_col) in DATASETS.items():
    r, templates = results[name], data[name][1]
    fit, _, test = r["parts"]
    background = fit[r["event_cols"]].sample(100, random_state=42) if name == "BGL" else None
    alerts[name] = example_alerts(r["model"], test, r["event_cols"], unseen_masks[name], name, templates, id_col,
                                  background=background)
    for a in alerts[name]:
        a["model"] = f"Random Forest ({r['method']})"
    path = PROCESSED / f"alerts_examples_{prefix}.json"
    save_alerts(alerts[name], path)
    reasons = pd.Series([a["reason"] for a in alerts[name]]).value_counts().to_dict()
    print(f"{path.relative_to(ROOT)}: {len(alerts[name])} alerts {reasons}")

data/processed/alerts_examples_hdfs.json: 20 alerts {'model': 18, 'unseen event': 2}


data/processed/alerts_examples_bgl.json: 20 alerts {'model': 15, 'unseen event': 5}


One model alert and one unseen-event alert per dataset:

In [7]:
for name in DATASETS:
    for reason in ["model", "unseen event"]:
        example = next((a for a in alerts[name] if a["reason"] == reason), None)
        if example:
            print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "window_id": "blk_-2551530448419944298",
  "dataset": "HDFS",
  "model": "Random Forest (balanced)",
  "anomaly_probability": 1.0,
  "reason": "model",
  "top_events": [
    {
      "event": "E4",
      "template": "Received block <BLK> of size <NUM> from /<IP>",
      "count": 0,
      "shap": 0.2089
    },
    {
      "event": "E1",
      "template": "Receiving block <BLK> src: /<IP> dest: /<IP>",
      "count": 2,
      "shap": 0.1071
    },
    {
      "event": "E5",
      "template": "BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to <BLK> size <NUM>",
      "count": 0,
      "shap": 0.0889
    }
  ]
}
{
  "window_id": "blk_6647747942787068087",
  "dataset": "HDFS",
  "model": "Random Forest (balanced)",
  "anomaly_probability": 0.37,
  "reason": "unseen event",
  "top_events": [
    {
      "event": "E44",
      "template": "Adding an already existing block <BLK>",
      "count": 2,
      "shap": null
    }
  ]
}
{
  "window_id": 166868,
  "dataset": "BGL",